# 01. 정상 거래와 무패턴 자금세탁 데이터 만들기

기존 `code/01_prepare_parquet.ipynb`의 결과를 사용하여 **패턴 자금세탁 거래만 제외**합니다. 패턴 연결은 `code/sampling_lib.py`와 동일하게 `edge_id`를 사용합니다.

| 구분 | 기준 | 출력 포함 |
|---|---|---|
| 정상 | `is_laundering = False` | 포함 |
| 무패턴 자금세탁 | `is_laundering = True`이며 `patterns.parquet`에 연결되지 않음 | 포함 |
| 패턴 자금세탁 | `is_laundering = True`이며 `patterns.parquet`에 연결됨 | 제외 |

여기서 **무패턴**은 `Patterns.txt`에 연결된 패턴 정보가 없다는 뜻이며, 거래 행동에 패턴이 전혀 없다는 뜻은 아닙니다.

- 전체 기간·통화·결제 방식을 유지합니다. 정상 샘플링이나 피처 계산은 하지 않습니다.
- 기존 거래 컬럼과 `edge_id`, `src`, `dst`를 그대로 보존합니다. 정답은 기존 `is_laundering`입니다.
- 입력은 이미 완전 중복을 제거한 거래입니다. 원본 CSV의 정상 179,476,683건과 달리 정상 **179,476,533건**이 예상됩니다.
- 정상 **179,476,533건** + 무패턴 자금세탁 **87,610건** = 출력 **179,564,143건**입니다.
- 이후 모델 입력에서 `is_laundering`은 제외하고 정답으로만 사용합니다.

## 0. 실행 위치와 입력

**GPU 서버의 Jupyter**에서 이 노트북을 위에서부터 실행합니다. 저장소 루트는 `/workspace/money_laundry_ai_/`이며, 기존 전처리의 아래 두 파일이 필요합니다. 서버 실행은 사용자가 직접 합니다.

```text
parquets/HI-Large/transactions.parquet
parquets/HI-Large/patterns.parquet
```

두 파일은 **같은 `01_prepare_parquet.ipynb` 실행 결과**여야 합니다. 패턴 파일이 없거나 매칭에 실패한 행이 있으면 중단합니다. 기존 전처리 결과가 없다면 먼저 `code/01_prepare_parquet.ipynb`를 실행합니다.

전체 거래를 pandas로 읽지 않고 DuckDB로 스캔·저장합니다. 출력은 정상 거래 대부분을 포함하므로 입력 거래 Parquet와 비슷한 규모의 디스크 공간이 필요합니다.

In [1]:
# 현재 Jupyter 커널에 패키지가 없을 때만 주석을 해제해 실행합니다.
!pip install --break-system-packages --no-cache-dir -q torch_geometric duckdb pandas

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json
import time
import duckdb
from IPython.display import display

DATASET = "HI-Large"
PROJECT_ROOT = None  # 자동 탐색 실패 시 Path("/workspace/money_laundry_ai_")
THREADS = 4
MEMORY = "6GB"      # 기존 환경의 컨테이너 메모리 한도를 고려해 설정

# 기존 HI-Large 전체 전처리 결과의 확인값. 다른 데이터셋은 별도 설정합니다.
EXPECTED_COUNTS = {
    "input_rows": 179_702_079,
    "normal_rows": 179_476_533,
    "laundering_rows": 225_546,
    "pattern_rows": 137_936,
    "nonpattern_laundering_rows": 87_610,
    "output_rows": 179_564_143,
}


def find_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "parquets" / DATASET / "transactions.parquet").is_file():
            return candidate
    raise FileNotFoundError(
        "기존 transactions.parquet를 찾지 못했습니다. "
        "PROJECT_ROOT를 지정하거나 먼저 01_prepare_parquet.ipynb를 실행하세요."
    )


ROOT = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else find_root()
BASE = ROOT / "parquets" / DATASET
OUT = BASE / "nonpaten_vs_normal"
TX_PATH = BASE / "transactions.parquet"
PATTERN_PATH = BASE / "patterns.parquet"
OUTPUT_PATH = OUT / "transactions.parquet"
SUMMARY_PATH = OUT / "summary.json"

for path in (TX_PATH, PATTERN_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)
for path in (OUTPUT_PATH, SUMMARY_PATH):
    if path.exists():
        raise FileExistsError(f"기존 결과를 덮어쓰지 않습니다. OUT을 새 폴더로 지정하세요: {path}")
print("입력:", BASE)
print("출력:", OUT)

입력: /workspace/money_laundry_ai_/parquets/HI-Large
출력: /workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal


## 1. 검증·추출 함수

패턴 파일의 미매칭·빈 `edge_id`·중복 연결, 거래 파일에 없는 ID, 정상 라벨에 연결된 패턴, 시각 불일치를 검사합니다. 입력 전체의 건수와 라벨별 건수도 확인합니다.

검증을 통과한 패턴 거래 ID만 제외하며, 기존 거래 파일을 변경하지 않습니다. 출력 행 순서는 보장하지 않으므로 이후 시간 분석에서는 `ts`와 `edge_id`로 정렬합니다.

In [3]:
def q(value):
    return "'" + str(value).replace("'", "''") + "'"


def require(condition, message):
    if not condition:
        raise ValueError(message)


def inspect_inputs(con, tx_path, pattern_path, expected=None):
    con.execute(f"CREATE OR REPLACE VIEW tx AS SELECT * FROM read_parquet({q(tx_path)})")
    con.execute(f"CREATE OR REPLACE VIEW pattern_source AS SELECT * FROM read_parquet({q(pattern_path)})")
    tx_schema = {r[0]: r[1] for r in con.execute("DESCRIBE tx").fetchall()}
    pat_schema = {r[0]: r[1] for r in con.execute("DESCRIBE pattern_source").fetchall()}
    require({"edge_id", "ts", "day", "is_laundering"} <= tx_schema.keys(),
            "transactions.parquet에 필수 컬럼이 없습니다.")
    require({"edge_id", "ts", "unmatched"} <= pat_schema.keys(),
            "patterns.parquet에 필수 컬럼이 없습니다. 01 전처리 결과를 확인하세요.")
    require(tx_schema["is_laundering"] == "BOOLEAN", "01 전처리의 BOOLEAN 라벨을 사용해야 합니다.")
    require(pat_schema["unmatched"] == "BOOLEAN", "unmatched는 BOOLEAN이어야 합니다.")

    pattern_n, bad_pattern_n, unique_pattern_n = con.execute("""
        SELECT count(*),
               count(*) FILTER (WHERE edge_id IS NULL OR ts IS NULL OR unmatched IS DISTINCT FROM FALSE),
               count(DISTINCT edge_id)
        FROM pattern_source
    """).fetchone()
    require(pattern_n > 0, "패턴 파일이 비어 있습니다. 전처리 완료 여부를 확인하세요.")
    require(bad_pattern_n == 0, f"미매칭 또는 필수값이 없는 패턴 행: {bad_pattern_n:,}")
    require(pattern_n == unique_pattern_n, "패턴 edge_id가 중복됩니다. 다중 매칭 여부를 확인하세요.")
    con.execute("CREATE OR REPLACE TEMP TABLE pattern_ids AS SELECT edge_id, ts FROM pattern_source")

    # 패턴은 소규모이므로 이 테이블을 기준으로 거래 파일을 대조합니다.
    joined_n, missing_n, wrong_label_n, wrong_ts_n = con.execute("""
        SELECT count(*),
               count(*) FILTER (WHERE t.edge_id IS NULL),
               count(*) FILTER (WHERE t.edge_id IS NOT NULL AND t.is_laundering IS DISTINCT FROM TRUE),
               count(*) FILTER (WHERE t.edge_id IS NOT NULL AND t.ts IS DISTINCT FROM p.ts)
        FROM pattern_ids p LEFT JOIN tx t ON t.edge_id = p.edge_id
    """).fetchone()
    require(missing_n == 0, f"거래 파일에서 찾지 못한 패턴 ID: {missing_n:,}")
    require(joined_n == pattern_n, "패턴 ID 하나에 거래가 여러 건 연결됩니다.")
    require(wrong_label_n == 0, f"양성 라벨이 아닌 거래에 연결된 패턴: {wrong_label_n:,}")
    require(wrong_ts_n == 0, f"패턴과 거래의 시각 불일치: {wrong_ts_n:,}. 같은 실행 결과인지 확인하세요.")

    total, normal, positive, invalid, first_ts, last_ts = con.execute("""
        SELECT count(*), count(*) FILTER (WHERE is_laundering = FALSE),
               count(*) FILTER (WHERE is_laundering = TRUE),
               count(*) FILTER (WHERE edge_id IS NULL OR ts IS NULL OR day IS NULL
                               OR is_laundering IS NULL OR day IS DISTINCT FROM CAST(ts AS DATE)),
               min(ts), max(ts)
        FROM tx
    """).fetchone()
    require(total > 0 and invalid == 0, f"입력 거래가 비었거나 필수값·날짜가 잘못되었습니다: {invalid:,}")
    stats = {
        "input_rows": total, "normal_rows": normal, "laundering_rows": positive,
        "pattern_rows": pattern_n, "nonpattern_laundering_rows": positive - pattern_n,
        "output_rows": total - pattern_n,
        "input_ts_min": str(first_ts), "input_ts_max": str(last_ts),
    }
    require(stats["nonpattern_laundering_rows"] >= 0, "패턴 거래 수가 전체 양성보다 많습니다.")
    if expected is not None:
        differences = {key: {"expected": value, "actual": stats.get(key)}
                       for key, value in expected.items() if stats.get(key) != value}
        require(not differences, f"기존 전처리 확인값과 다릅니다: {differences}")
    return stats

In [4]:
def export_dataset(con, output_path, summary_path, stats, metadata):
    output_path, summary_path = Path(output_path), Path(summary_path)
    require(output_path != summary_path, "데이터와 요약 파일 경로가 같습니다.")
    for path in (output_path, summary_path):
        if path.exists():
            raise FileExistsError(f"기존 결과를 덮어쓰지 않습니다: {path}")
        path.parent.mkdir(parents=True, exist_ok=True)
    token = uuid4().hex
    staged_data = output_path.with_name(f".{output_path.stem}.{token}.partial.parquet")
    staged_summary = summary_path.with_name(f".{summary_path.stem}.{token}.partial.json")
    try:
        con.execute(f"""
            COPY (
                SELECT t.* FROM tx t
                ANTI JOIN pattern_ids p
                  ON t.edge_id = p.edge_id AND t.is_laundering = TRUE
            ) TO {q(staged_data)} (FORMAT PARQUET, COMPRESSION ZSTD)
        """)
        con.execute(f"CREATE OR REPLACE VIEW result AS SELECT * FROM read_parquet({q(staged_data)})")
        n, normal, positive, first_ts, last_ts = con.execute("""
            SELECT count(*), count(*) FILTER (WHERE is_laundering = FALSE),
                   count(*) FILTER (WHERE is_laundering = TRUE), min(ts), max(ts)
            FROM result
        """).fetchone()
        require((n, normal, positive) == (stats["output_rows"], stats["normal_rows"],
                                         stats["nonpattern_laundering_rows"]),
                f"출력 건수 불일치: 전체={n:,}, 정상={normal:,}, 무패턴={positive:,}")
        remaining = con.execute("""
            SELECT count(*) FROM result r JOIN pattern_ids p ON r.edge_id = p.edge_id
            WHERE r.is_laundering = TRUE
        """).fetchone()[0]
        require(remaining == 0, f"출력에 패턴 자금세탁이 남았습니다: {remaining:,}")
        require(con.execute("DESCRIBE tx").fetchall() == con.execute("DESCRIBE result").fetchall(),
                "입력과 출력의 컬럼 또는 자료형이 다릅니다.")
        result = {
            **metadata, **stats,
            "output_ts_min": str(first_ts), "output_ts_max": str(last_ts),
            "remaining_pattern_laundering_rows": remaining,
            "output_file": str(output_path), "output_bytes": staged_data.stat().st_size,
            "label_column": "is_laundering",
            "label_meaning": {"false": "normal", "true": "nonpattern_laundering"},
            "duplicate_policy": "preserve_input (01 preprocessing already removed full duplicates)",
            "sampling": "none", "date_filter": None,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        staged_summary.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        con.execute("DROP VIEW result")
        # 검증된 파일만 최종 이름으로 공개합니다. summary.json은 완료 후 마지막에 저장합니다.
        staged_data.replace(output_path)
        staged_summary.replace(summary_path)
        return result
    finally:
        # 이번 실행에서 만든 임시 파일만 정리합니다.
        con.execute("DROP VIEW IF EXISTS result")
        staged_data.unlink(missing_ok=True)
        staged_summary.unlink(missing_ok=True)

## 2. 검증 후 데이터 파일 저장

아래 셀이 전체 데이터를 읽고 파일을 저장합니다. 메모리 한도는 `MEMORY`, 스캔 병렬도는 `THREADS`로 조정합니다. 기존 결과가 있으면 덮어쓰지 않고 중단합니다.

In [5]:
OUT.mkdir(parents=True, exist_ok=True)
spill_dir = OUT / "duckdb_tmp" / uuid4().hex
spill_dir.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
t0 = time.perf_counter()
try:
    con.execute(f"SET threads={int(THREADS)}")
    con.execute(f"SET memory_limit={q(MEMORY)}")
    con.execute(f"SET temp_directory={q(spill_dir)}")
    con.execute("SET preserve_insertion_order=false")
    print("입력과 패턴 매칭 검증 중...", flush=True)
    stats = inspect_inputs(con, TX_PATH, PATTERN_PATH, EXPECTED_COUNTS)
    print(json.dumps(stats, ensure_ascii=False, indent=2), flush=True)
    print("정상 + 무패턴 자금세탁 Parquet 저장 및 재검증 중...", flush=True)
    summary = export_dataset(con, OUTPUT_PATH, SUMMARY_PATH, stats, {
        "dataset": DATASET,
        "input_transactions": str(TX_PATH), "input_patterns": str(PATTERN_PATH),
        "input_transactions_bytes": TX_PATH.stat().st_size,
        "input_patterns_bytes": PATTERN_PATH.stat().st_size,
        "input_transactions_mtime_ns": TX_PATH.stat().st_mtime_ns,
        "input_patterns_mtime_ns": PATTERN_PATH.stat().st_mtime_ns,
        "duckdb_version": duckdb.__version__,
    })
    print(f"완료: {time.perf_counter() - t0:,.1f}초")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
finally:
    con.close()
    if spill_dir.is_dir() and not any(spill_dir.iterdir()):
        spill_dir.rmdir()

입력과 패턴 매칭 검증 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "input_rows": 179702079,
  "normal_rows": 179476533,
  "laundering_rows": 225546,
  "pattern_rows": 137936,
  "nonpattern_laundering_rows": 87610,
  "output_rows": 179564143,
  "input_ts_min": "2022-08-01 00:00:00",
  "input_ts_max": "2023-01-12 16:49:00"
}
정상 + 무패턴 자금세탁 Parquet 저장 및 재검증 중...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

완료: 12,202.3초
{
  "dataset": "HI-Large",
  "input_transactions": "/workspace/money_laundry_ai_/parquets/HI-Large/transactions.parquet",
  "input_patterns": "/workspace/money_laundry_ai_/parquets/HI-Large/patterns.parquet",
  "input_transactions_bytes": 4799016434,
  "input_patterns_bytes": 1336195,
  "input_transactions_mtime_ns": 1792476370073602157,
  "input_patterns_mtime_ns": 1792477967698810144,
  "duckdb_version": "1.5.6",
  "input_rows": 179702079,
  "normal_rows": 179476533,
  "laundering_rows": 225546,
  "pattern_rows": 137936,
  "nonpattern_laundering_rows": 87610,
  "output_rows": 179564143,
  "input_ts_min": "2022-08-01 00:00:00",
  "input_ts_max": "2023-01-12 16:49:00",
  "output_ts_min": "2022-08-01 00:00:00",
  "output_ts_max": "2023-01-12 16:49:00",
  "remaining_pattern_laundering_rows": 0,
  "output_file": "/workspace/money_laundry_ai_/parquets/HI-Large/nonpaten_vs_normal/transactions.parquet",
  "output_bytes": 4820735859,
  "label_column": "is_laundering",
  "label_m

## 3. 저장 결과 확인

```text
parquets/HI-Large/nonpaten_vs_normal/
├─ transactions.parquet   정상 + 무패턴 자금세탁, 기존 거래 컬럼 유지
└─ summary.json           입력·제외·출력 건수 및 검증 결과
```

전체를 DataFrame으로 불러오지 않고 라벨별 집계와 각 5행만 확인합니다. `is_laundering=False`는 정상, `True`는 무패턴 자금세탁입니다.

In [6]:
with duckdb.connect() as check:
    check.execute(f"SET memory_limit={q(MEMORY)}")
    check.execute(f"CREATE VIEW selected AS SELECT * FROM read_parquet({q(OUTPUT_PATH)})")
    display(check.execute("""
        SELECT is_laundering,
               CASE WHEN is_laundering THEN 'nonpattern_laundering' ELSE 'normal' END AS class_name,
               count(*) AS rows, min(ts) AS first_ts, max(ts) AS last_ts
        FROM selected GROUP BY 1 ORDER BY 1
    """).df())
    for label in (False, True):
        print("무패턴 자금세탁" if label else "정상")
        display(check.execute(f"SELECT * FROM selected WHERE is_laundering = {str(label).upper()} LIMIT 5").df())

,is_laundering,class_name,rows,first_ts,last_ts
0,False,normal,179476533,2022-08-01 00:00:00,2023-01-12 16:49:00
1,True,nonpattern_laundering,87610,2022-08-01 00:01:00,2022-11-05 23:59:00


정상


,edge_id,ts,src,dst,paid,received,payment_currency,receiving_currency,payment_format,duplicate_group_id,is_laundering,from_bank,from_account,to_bank,to_account,day
0,1,2022-08-01,3,3,19.18,19.18,US Dollar,US Dollar,Reinvestment,<NA>,False,0,80006DFE0,0,80006DFE0,2022-08-01
1,2,2022-08-01,25,78803,17955.11,17955.11,US Dollar,US Dollar,Cheque,<NA>,False,0,800077B00,867,80079A2C0,2022-08-01
2,3,2022-08-01,31,86028,1199.83,1199.83,US Dollar,US Dollar,Cash,<NA>,False,0,80010E900,1208,80011CC20,2022-08-01
3,4,2022-08-01,39,269980,148565.46,148565.46,US Dollar,US Dollar,Wire,<NA>,False,0,800117F00,16315,8040DF950,2022-08-01
4,5,2022-08-01,50,29866,227247.42,227247.42,US Dollar,US Dollar,Cheque,<NA>,False,0,8001202E0,20,800226BD0,2022-08-01


무패턴 자금세탁


,edge_id,ts,src,dst,paid,received,payment_currency,receiving_currency,payment_format,duplicate_group_id,is_laundering,from_bank,from_account,to_bank,to_account,day
0,45868,2022-08-01 00:01:00,47315,647714,106901.34,106901.34,Canadian Dollar,Canadian Dollar,Cheque,<NA>,True,70,100428858,96670,82838D2B0,2022-08-01
1,45960,2022-08-01 00:01:00,47319,1888383,26458.93,26458.93,Swiss Franc,Swiss Franc,Cash,<NA>,True,70,100428978,2173061,83FB5E550,2022-08-01
2,132567,2022-08-01 00:03:00,47313,1017357,16201.55,16201.55,Ruble,Ruble,Credit Card,<NA>,True,70,1004287C8,186057,821AC59A0,2022-08-01
3,144428,2022-08-01 00:03:00,532946,1047199,4663.02,4663.02,US Dollar,US Dollar,ACH,<NA>,True,39921,837F1BEB0,195762,837F1C360,2022-08-01
4,201914,2022-08-01 00:04:00,1128877,84880,1674.49,1674.49,Euro,Euro,ACH,<NA>,True,218695,80BAB6B60,1148,80BAB7220,2022-08-01
